In [174]:
import requests, json
from pathlib import Path

SRC = Path("../sources").resolve()

overpass_url = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:180];
area[name="Berlin"]["boundary"="administrative"]->.searchArea;

(
  way["natural"="water"]["water"~"lake|pond"](area.searchArea);
  relation["natural"="water"]["water"~"lake|pond"](area.searchArea);
);
out body;
>;
out skel qt;
"""

print("Fetching OSM lakes from Overpass...")
resp = requests.get(overpass_url, params={"data": query})

if resp.status_code == 200:
    data = resp.json()
    out_path = SRC / "osm_berlin_lakes_raw.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f)
    print("✅ OSM raw data saved to:", out_path)
else:
    print("❌ Error:", resp.status_code)
    print(resp.text[:300])


Fetching OSM lakes from Overpass...
✅ OSM raw data saved to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json


In [177]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

SRC = Path("../sources").resolve()
csv_path = SRC / "berlin_lakes_summary.csv"
geo_path = SRC / "berlin_lakes_clean.geojson"

print("Reading CSV from:", csv_path)
df = pd.read_csv(csv_path, sep=";")

print("CSV shape:", df.shape)
print("Columns:", df.columns.tolist())

# --- make sure coordinate columns are present ---
# if your file had "lat"/"lon" instead, we would rename them
if "centroid_lon" not in df.columns and "lon" in df.columns:
    df = df.rename(columns={"lon": "centroid_lon"})
if "centroid_lat" not in df.columns and "lat" in df.columns:
    df = df.rename(columns={"lat": "centroid_lat"})

coord_cols = ["centroid_lon", "centroid_lat"]
missing = [c for c in coord_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing coordinate columns: {missing}")

# drop rows without coordinates (should still leave many lakes)
before = df.shape[0]
df = df.dropna(subset=coord_cols)
print(f"After dropping rows without coordinates: {df.shape} "
      f"(removed {before - df.shape[0]} rows)")

# --- build GeoDataFrame from centroids ---
gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["centroid_lon"], df["centroid_lat"]),
    crs="EPSG:4326",
)

print("GeoDataFrame shape:", gdf.shape)

# --- save GeoJSON; DOES NOT touch other files ---
gdf.to_file(geo_path, driver="GeoJSON")
print("Saved GeoJSON:", geo_path)


Reading CSV from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_summary.csv
CSV shape: (276, 12)
Columns: ['lake_name', 'centroid_lat', 'centroid_lon', 'water_type', 'area_ha', 'max_depth_m', 'perimeter_m', 'has_public_access', 'swimming_allowed', 'water_quality', 'data_source', 'last_updated']
After dropping rows without coordinates: (276, 12) (removed 0 rows)
GeoDataFrame shape: (276, 13)
Saved GeoJSON: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson


In [181]:
%pip install -q --upgrade pandas geopandas pyogrio shapely fiona


Note: you may need to restart the kernel to use updated packages.


In [183]:
import os, glob, pathlib
SRC = pathlib.Path("../sources").resolve()
print("Looking in:", SRC)
files = sorted(glob.glob(str(SRC / "*")))
for f in files:
    print("-", os.path.basename(f))


Looking in: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources
- README.md
- berlin_lakes_clean.geojson
- berlin_lakes_summary.csv
- daemeritzsee_polygon.geojson
- demeritzsee.csv
- demeritzsee_clean.csv
- lakes_berlin_unified.geojson
- osm_berlin_lakes.geojson
- osm_berlin_lakes_clean.geojson
- osm_berlin_lakes_raw.json
- scripts


In [186]:
from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()
# use the cleaned GeoJSON that we just created
geo_path = SRC / "berlin_lakes_clean.geojson"

print("Looking for:", geo_path)
print("Exists:", geo_path.exists())

if not geo_path.exists():
    raise FileNotFoundError(f"Cannot find file at: {geo_path}")

gdf = gpd.read_file(geo_path)
print("✅ GeoDataFrame loaded")
print("Rows:", len(gdf))
print("Columns:", list(gdf.columns))
print("CRS:", gdf.crs)
gdf.head(3)


Looking for: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson
Exists: True
✅ GeoDataFrame loaded
Rows: 276
Columns: ['lake_name', 'centroid_lat', 'centroid_lon', 'water_type', 'area_ha', 'max_depth_m', 'perimeter_m', 'has_public_access', 'swimming_allowed', 'water_quality', 'data_source', 'last_updated', 'geometry']
CRS: EPSG:4326


,lake_name,centroid_lat,centroid_lon,water_type,area_ha,max_depth_m,perimeter_m,has_public_access,swimming_allowed,water_quality,data_source,last_updated,geometry
0,jungfernheideteich,52.543856,13.278930,pond,6.8416,None,None,None,None,None,OSM waterbodies (Berlin),2025-11-12 10:22:47.569,POINT (13.27893 52.54386)
1,spandauer see,52.550213,13.219207,lake,107.7303,None,None,None,None,None,OSM waterbodies (Berlin),2025-11-12 10:22:47.569,POINT (13.21921 52.55021)
2,hubertussee,52.486131,13.280684,lake,2.4239,None,None,None,None,None,OSM waterbodies (Berlin),2025-11-12 10:22:47.569,POINT (13.28068 52.48613)


In [190]:
# Reloading the CSV with semicolon separator
df = pd.read_csv(csv_path, sep=";", skiprows=5)

print("✅ CSV reloaded:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)


✅ CSV reloaded: (271, 12)
Columns: ['langer see', '52.400747119279224', '13.625819492700556', 'lake', '312.7947', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'OSM waterbodies (Berlin)', '2025-11-12 10:22:47.569398']


,langer see,52.400747119279224,13.625819492700556,lake,312.7947,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
0,neuer see,52.511383,13.342410,pond,4.6690,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
1,schafersee,52.564429,13.360881,lake,4.1906,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
2,springpfuhl,52.529139,13.540301,lake,1.2979,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
3,groß glienicker see,52.465374,13.112259,lake,63.9042,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
4,havel,52.436294,13.127598,lake,337.7410,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
5,obersee,52.548601,13.489121,lake,3.7330,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
6,zeuthener see,52.356933,13.641020,lake,213.4706,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
7,steinbergsee,52.597672,13.308929,lake,0.9645,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
8,fauler see,52.519284,13.212941,lake,2.1215,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
9,hauptsee,52.434277,13.417135,pond,5.2876,NaN,NaN,NaN,NaN,NaN,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398


In [200]:
# --- Safely drop meta/header rows and empty columns ---

# Some rows in "dt" contain text like "water physics" or "YYYY-MM-DD".
# We only try this if a suitable time column exists.
time_col = None
if "dt" in df.columns:
    time_col = "dt"
elif "datetime" in df.columns:
    time_col = "datetime"

if time_col is not None:
    # Identify meta/header rows
    mask_meta = df[time_col].astype(str).str.contains(
        "water physics|YYYY", na=False
    )
    removed = mask_meta.sum()
    rows_before = len(df)

    # Keep only non-meta rows
    df = df[~mask_meta].copy()

    print(f"✅ Removed {removed} meta/header rows "
          f"(from {rows_before} to {len(df)}).")
else:
    print("ℹ️ No 'dt' or 'datetime' column found – skipping meta row removal.")

# Drop completely empty columns (this is safe)
df = df.dropna(axis=1, how="all")

print("✅ Cleaned headers, remaining rows:", df.shape)
print("Columns now:", df.columns.tolist())
df.head(10)


ℹ️ No 'dt' or 'datetime' column found – skipping meta row removal.
✅ Cleaned headers, remaining rows: (271, 7)
Columns now: ['langer see', '52.400747119279224', '13.625819492700556', 'lake', '312.7947', 'OSM waterbodies (Berlin)', '2025-11-12 10:22:47.569398']


,langer see,52.400747119279224,13.625819492700556,lake,312.7947,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
0,neuer see,52.511383,13.342410,pond,4.6690,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
1,schafersee,52.564429,13.360881,lake,4.1906,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
2,springpfuhl,52.529139,13.540301,lake,1.2979,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
3,groß glienicker see,52.465374,13.112259,lake,63.9042,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
4,havel,52.436294,13.127598,lake,337.7410,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
5,obersee,52.548601,13.489121,lake,3.7330,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
6,zeuthener see,52.356933,13.641020,lake,213.4706,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
7,steinbergsee,52.597672,13.308929,lake,0.9645,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
8,fauler see,52.519284,13.212941,lake,2.1215,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
9,hauptsee,52.434277,13.417135,pond,5.2876,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398


In [204]:
rename_map = {
    'dt': 'datetime',
    'w_temp': 'temp_c',
    'cond': 'conductivity_µS',
    'depth': 'depth_m',
    'pH': 'pH',
    'o2_sat': 'oxygen_saturation_pct',
    'o2_con': 'oxygen_concentration_mgL',
    'secchi': 'secchi_depth_m'
}
df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})
print("✅ Columns renamed:")
df.head(3)


✅ Columns renamed:


,langer see,52.400747119279224,13.625819492700556,lake,312.7947,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
0,neuer see,52.511383,13.342410,pond,4.6690,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
1,schafersee,52.564429,13.360881,lake,4.1906,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398
2,springpfuhl,52.529139,13.540301,lake,1.2979,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398


In [211]:
from pathlib import Path
import pandas as pd

SRC = Path("../sources").resolve()
raw_path = SRC / "demeritzsee.csv"
out_csv = SRC / "demeritzsee_clean.csv"

# 1) Load raw file – skip the metadata lines at the top
df_dem = pd.read_csv(raw_path, sep=";", skiprows=5)

print("Raw Dämmeritzsee loaded:", df_dem.shape)
print("Columns:", df_dem.columns.tolist())

# 2) Keep only real measurement rows (dt starts with a date like 1992-04-30 …)
mask_measurements = df_dem["dt"].astype(str).str.match(r"\d{4}-\d{2}-\d{2}", na=False)
df_dem = df_dem[mask_measurements].copy()
print("After keeping only measurements:", df_dem.shape)

# 3) Rename columns to nice names
rename_map = {
    "dt": "datetime",
    "w_temp": "temp_c",
    "cond": "conductivity_µS",
    "depth": "depth_m",
    "pH": "pH",
    "o2_sat": "oxygen_saturation_pct",
    "o2_con": "oxygen_concentration_mgL",
    "secchi": "secchi_depth_m",
}
df_dem = df_dem.rename(columns={c: rename_map.get(c, c) for c in df_dem.columns})
print("Columns renamed:", df_dem.columns.tolist())

# 4) Convert datetime and numeric columns
df_dem["datetime"] = pd.to_datetime(df_dem["datetime"], errors="coerce")
df_dem = df_dem[df_dem["datetime"].notna()].copy()

numeric_cols = [
    "temp_c",
    "conductivity_µS",
    "depth_m",
    "pH",
    "oxygen_saturation_pct",
    "oxygen_concentration_mgL",
    "secchi_depth_m",
]
for col in numeric_cols:
    if col in df_dem.columns:
        df_dem[col] = pd.to_numeric(df_dem[col], errors="coerce")

print("Converted types:")
print(df_dem.dtypes)

# 5) Add lake_name and source columns
df_dem["lake_name"] = "Dämeritzsee"
df_dem["source"] = "In-situ measurement"

print("Clean DataFrame ready:", df_dem.shape)
df_dem.head(5)

# 6) Save clean CSV
df_dem.to_csv(out_csv, index=False, sep=";")
print("✅ Saved clean Dämmeritzsee CSV to:", out_csv)


Raw Dämmeritzsee loaded: (198, 9)
Columns: ['dt', 'w_temp', 'cond', 'depth', 'pH', 'o2_sat', 'o2_con', 'secchi', 'Unnamed: 8']
After keeping only measurements: (194, 9)
Columns renamed: ['datetime', 'temp_c', 'conductivity_µS', 'depth_m', 'pH', 'oxygen_saturation_pct', 'oxygen_concentration_mgL', 'secchi_depth_m', 'Unnamed: 8']
Converted types:
datetime                    datetime64[ns]
temp_c                             float64
conductivity_µS                    float64
depth_m                            float64
pH                                 float64
oxygen_saturation_pct              float64
oxygen_concentration_mgL           float64
secchi_depth_m                     float64
Unnamed: 8                          object
dtype: object
Clean DataFrame ready: (194, 11)
✅ Saved clean Dämmeritzsee CSV to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/demeritzsee_clean.csv


In [217]:
df['lake_name'] = "Dämeritzsee"
df['source'] = "In-situ measurement"

print("✅ Clean DataFrame ready:", df.shape)
df.head(5)


✅ Clean DataFrame ready: (271, 9)


,langer see,52.400747119279224,13.625819492700556,lake,312.7947,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398,lake_name,source
0,neuer see,52.511383,13.342410,pond,4.6690,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398,Dämeritzsee,In-situ measurement
1,schafersee,52.564429,13.360881,lake,4.1906,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398,Dämeritzsee,In-situ measurement
2,springpfuhl,52.529139,13.540301,lake,1.2979,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398,Dämeritzsee,In-situ measurement
3,groß glienicker see,52.465374,13.112259,lake,63.9042,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398,Dämeritzsee,In-situ measurement
4,havel,52.436294,13.127598,lake,337.7410,OSM waterbodies (Berlin),2025-11-12 10:22:47.569398,Dämeritzsee,In-situ measurement


In [225]:
clean_csv_path = Path("../sources/demeritzsee_clean.csv")
df.to_csv(clean_csv_path, index=False)
print("💾 Saved cleaned dataset to:", clean_csv_path)


💾 Saved cleaned dataset to: ../sources/demeritzsee_clean.csv


In [228]:
import geopandas as gpd
from pathlib import Path

geo_path = Path("../sources/osm_berlin_lakes.geojson")

print("Looking for:", geo_path)
print("Exists:", geo_path.exists())

if not geo_path.exists():
    raise FileNotFoundError(f"Cannot find file at: {geo_path}")

gdf = gpd.read_file(geo_path)

print("GeoDataFrame loaded:", gdf.shape)
print("Columns:", list(gdf.columns))
print("CRS:", gdf.crs)
gdf.head(3)


Looking for: ../sources/osm_berlin_lakes.geojson
Exists: True
GeoDataFrame loaded: (1908, 125)
Columns: ['id', '@id', 'NHD:ComID', 'NHD:Elevation', 'NHD:FCode', 'NHD:FDate', 'NHD:FTYPE', 'NHD:Permanent_', 'NHD:ReachCode', 'NHD:Resolution', 'TMC:cid_58:tabcd_1:Class', 'TMC:cid_58:tabcd_1:LCLversion', 'TMC:cid_58:tabcd_1:LocationCode', 'access', 'addr:housenumber', 'addr:street', 'alt_name', 'amenity', 'artist_name', 'artwork_type', 'attraction', 'barrier', 'basin', 'bathing', 'boat', 'boundary', 'building:part', 'canoe', 'communication:amateur_radio:pota', 'construction:attraction', 'covered', 'created_by', 'description', 'designation', 'detention', 'drinking_water', 'ele', 'email', 'emergency', 'fishing', 'fixme', 'fountain', 'ft_link', 'genus:wikidata', 'gnis:feature_id', 'golf', 'height', 'historic', 'historic:water', 'image', 'intermittent', 'kerb', 'landcover', 'landuse', 'layer', 'leisure', 'level', 'lit', 'loc_name', 'loc_ref', 'man_made', 'mapillary', 'maxspeed', 'mooring', 'mot

,id,@id,NHD:ComID,NHD:Elevation,NHD:FCode,NHD:FDate,NHD:FTYPE,NHD:Permanent_,NHD:ReachCode,NHD:Resolution,...,water,waterway:name,website,wetland,wheelchair,width,wikidata,wikimedia_commons,wikipedia,geometry
0,relation/3217,relation/3217,None,None,None,NaT,None,None,None,None,...,pond,None,None,None,None,None,Q63887019,Category:Jungfernheideteich,None,"POLYGON ((13.27588 52.54329, 13.27594 52.54328..."
1,relation/3295,relation/3295,None,None,None,NaT,None,None,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((13.09516 52.41275, 13.09533 52.41267..."
2,relation/4026,relation/4026,None,None,None,NaT,None,None,None,None,...,lake,None,None,None,None,None,Q63284050,None,None,"POLYGON ((13.20902 52.54121, 13.20901 52.54125..."


In [230]:
# Keeping only key columns
cols_to_keep = ['name', 'natural', 'water', 'wikidata', 'geometry']

existing = [c for c in cols_to_keep if c in gdf.columns]
missing = [c for c in cols_to_keep if c not in gdf.columns]

print("Keeping:", existing)
print("Missing (ignored):", missing)

gdf_clean = gdf[existing].copy()

print("Cleaned OSM shape:", gdf_clean.shape)
gdf_clean.head(3)


Keeping: ['name', 'natural', 'water', 'wikidata', 'geometry']
Missing (ignored): []
Cleaned OSM shape: (1908, 5)


,name,natural,water,wikidata,geometry
0,Jungfernheideteich,water,pond,Q63887019,"POLYGON ((13.27588 52.54329, 13.27594 52.54328..."
1,Wiesenteich,water,None,None,"POLYGON ((13.09516 52.41275, 13.09533 52.41267..."
2,Spandauer See,water,lake,Q63284050,"POLYGON ((13.20902 52.54121, 13.20901 52.54125..."


In [136]:
# Keeping only key columns
cols_to_keep = ['name', 'natural', 'water', 'wikidata', 'geometry']
gdf = gdf[[c for c in cols_to_keep if c in gdf.columns]]

# Removeing unnamed water bodies (no name)
gdf = gdf[gdf['name'].notna()].reset_index(drop=True)

# Droping duplicates by name
gdf = gdf.drop_duplicates(subset='name')

print("✅ Cleaned GeoDataFrame:", gdf.shape)
gdf.head(10)


✅ Cleaned GeoDataFrame: (275, 2)


,name,geometry
0,jungfernheideteich,POINT (13.27893 52.54386)
1,spandauer see,POINT (13.21921 52.55021)
2,hubertussee,POINT (13.28068 52.48613)
3,seddinsee,POINT (13.681 52.38677)
4,langer see,POINT (13.62582 52.40075)
5,neuer see,POINT (13.34241 52.51138)
6,schafersee,POINT (13.36088 52.56443)
7,springpfuhl,POINT (13.5403 52.52914)
8,groß glienicker see,POINT (13.11226 52.46537)
9,havel,POINT (13.1276 52.43629)


In [234]:
clean_osm_path = Path("../sources/osm_berlin_lakes_clean.geojson")
gdf_clean.to_file(clean_osm_path, driver="GeoJSON")

print("💾 Saved cleaned OSM lakes to:", clean_osm_path)


💾 Saved cleaned OSM lakes to: ../sources/osm_berlin_lakes_clean.geojson


In [138]:
# Searchinf for Dämeritzsee in the OSM data
mask = gdf['name'].str.contains("Dämeritz", case=False, na=False)
gdf_lake = gdf[mask]

print("✅ Found matching lake(s):", len(gdf_lake))
gdf_lake


✅ Found matching lake(s): 0


,name,geometry


In [140]:
out_path = Path("../sources/daemeritzsee_polygon.geojson")
gdf_lake.to_file(out_path, driver="GeoJSON")
print("💾 Saved Dämeritzsee polygon to:", out_path)


💾 Saved Dämeritzsee polygon to: ../sources/daemeritzsee_polygon.geojson


In [239]:
# --- Clean + enrich OSM lakes (Berlin) ---

import unicodedata

# At this point `gdf` is already loaded from the previous cell.
# We'll work on a copy so we don't overwrite the original accidentally.
gdf = gdf.copy()

# keeping only essential columns (only those that actually exist)
cols_to_keep = [c for c in ["name", "natural", "water", "wikidata", "geometry"] if c in gdf.columns]
gdf = gdf[cols_to_keep].copy()

# keeping water bodies that are lakes/ponds (only if these columns exist)
if "natural" in gdf.columns:
    gdf = gdf[gdf["natural"].fillna("").str.lower().eq("water")]
if "water" in gdf.columns:
    gdf = gdf[gdf["water"].fillna("").str.lower().isin(["lake", "pond"])]

# unamed and duplicates – drop rows without a name, then deduplicate by name
if "name" in gdf.columns:
    gdf = gdf[gdf["name"].notna()].drop_duplicates(subset="name").reset_index(drop=True)

    # adding normalized name for robust matching (strip accents, lowercase)
    def normalize(s):
        s = unicodedata.normalize("NFKD", str(s))
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        return s.lower().strip()

    gdf["name_norm"] = gdf["name"].apply(normalize)

# project to metric CRS around Berlin for correct areas (EPSG:25833 is UTM33N)
gdf_metric = gdf.to_crs(25833)

# computing area (sq km) and centroids (lat/lon)
gdf_metric["area_sqkm"] = gdf_metric.geometry.area / 1e6
centroids_lonlat = gdf_metric.to_crs(4326).centroid
gdf_metric["centroid_lon"] = centroids_lonlat.x
gdf_metric["centroid_lat"] = centroids_lonlat.y

# bringing back to WGS84 for saving
gdf_clean = gdf_metric.to_crs(4326)

print("✅ Cleaned OSM gdf:", gdf_clean.shape)
gdf_clean.head()


✅ Cleaned OSM gdf: (276, 9)


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_4197/3050197729.py:36: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_lonlat = gdf_metric.to_crs(4326).centroid


,name,natural,water,wikidata,geometry,name_norm,area_sqkm,centroid_lon,centroid_lat
0,Jungfernheideteich,water,pond,Q63887019,"POLYGON ((13.27588 52.54329, 13.27594 52.54328...",jungfernheideteich,0.068416,13.278930,52.543856
1,Spandauer See,water,lake,Q63284050,"POLYGON ((13.20902 52.54121, 13.20901 52.54125...",spandauer see,1.077303,13.219207,52.550213
2,Hubertussee,water,lake,Q1616489,"POLYGON ((13.28369 52.48544, 13.28369 52.48545...",hubertussee,0.024239,13.280684,52.486131
3,Seddinsee,water,lake,Q2264279,"POLYGON ((13.70247 52.39344, 13.70268 52.39344...",seddinsee,2.624847,13.681003,52.386774
4,Langer See,water,lake,Q1805169,"POLYGON ((13.6158 52.40695, 13.61638 52.40712,...",langer see,3.127950,13.625820,52.400747


In [146]:
out_geo = Path("../sources/berlin_lakes_clean.geojson")
out_csv = Path("../sources/berlin_lakes_summary.csv")

# GeoJSON
gdf_clean[["name","name_norm","natural","water","wikidata","area_sqkm",
           "centroid_lon","centroid_lat","geometry"]].to_file(out_geo, driver="GeoJSON")

# Saving my CSV without geometry for easy viewing
gdf_clean.drop(columns="geometry").to_csv(out_csv, index=False)

print("💾 Saved:", out_geo)
print("💾 Saved:", out_csv)


💾 Saved: ../sources/berlin_lakes_clean.geojson
💾 Saved: ../sources/berlin_lakes_summary.csv


In [241]:
# --- Clean + enrich OSM lakes (Berlin) ---

import unicodedata

# `gdf` is already loaded from the OSM loader cell above.
# Work on a copy:
gdf = gdf.copy()

# keep only essential columns that actually exist
cols_to_keep = [c for c in ["name", "natural", "water", "wikidata", "geometry"] if c in gdf.columns]
gdf = gdf[cols_to_keep].copy()

# keep only water bodies that look like lakes/ponds
if "natural" in gdf.columns:
    gdf = gdf[gdf["natural"].fillna("").str.lower().eq("water")]

if "water" in gdf.columns:
    gdf = gdf[gdf["water"].fillna("").str.lower().isin(["lake", "pond"])]

# drop unnamed and duplicates
if "name" in gdf.columns:
    gdf = gdf[gdf["name"].notna()].drop_duplicates(subset="name").reset_index(drop=True)

    # normalize name
    def normalize(s):
        s = unicodedata.normalize("NFKD", str(s))
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        return s.lower().strip()

    gdf["name_norm"] = gdf["name"].apply(normalize)

# project to metric CRS for area + centroid computations
gdf_metric = gdf.to_crs(25833)

gdf_metric["area_sqkm"] = gdf_metric.geometry.area / 1e6

centroids = gdf_metric.to_crs(4326).centroid
gdf_metric["centroid_lon"] = centroids.x
gdf_metric["centroid_lat"] = centroids.y

# convert back to WGS84 for saving
gdf_clean = gdf_metric.to_crs(4326)

print("✅ Cleaned OSM gdf:", gdf_clean.shape)
gdf_clean.head()


✅ Cleaned OSM gdf: (276, 9)


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_4197/1454723452.py:37: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf_metric.to_crs(4326).centroid


,name,natural,water,wikidata,geometry,name_norm,area_sqkm,centroid_lon,centroid_lat
0,Jungfernheideteich,water,pond,Q63887019,"POLYGON ((13.27588 52.54329, 13.27594 52.54328...",jungfernheideteich,0.068416,13.278930,52.543856
1,Spandauer See,water,lake,Q63284050,"POLYGON ((13.20902 52.54121, 13.20901 52.54125...",spandauer see,1.077303,13.219207,52.550213
2,Hubertussee,water,lake,Q1616489,"POLYGON ((13.28369 52.48544, 13.28369 52.48545...",hubertussee,0.024239,13.280684,52.486131
3,Seddinsee,water,lake,Q2264279,"POLYGON ((13.70247 52.39344, 13.70268 52.39344...",seddinsee,2.624847,13.681003,52.386774
4,Langer See,water,lake,Q1805169,"POLYGON ((13.6158 52.40695, 13.61638 52.40712,...",langer see,3.127950,13.625820,52.400747


In [243]:
candidates = ["dämeritzsee", "daemeritzsee", "dameritzsee"]

mask = gdf_clean["name_norm"].apply(
    lambda s: any(v in s for v in candidates)
)

gdf_dameritz = gdf_clean[mask].copy()
print("🔎 matches:", len(gdf_dameritz))
gdf_dameritz[["name","area_sqkm","centroid_lon","centroid_lat"]].head(10)


🔎 matches: 0


,name,area_sqkm,centroid_lon,centroid_lat


In [150]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path("../sources/demeritzsee_clean.csv"), parse_dates=["datetime"])

stats = {
    "n_rows": len(df),
    "date_min": df["datetime"].min(),
    "date_max": df["datetime"].max(),
    "temp_c_mean": round(df["temp_c"].mean(), 2),
    "temp_c_min": round(df["temp_c"].min(), 2),
    "temp_c_max": round(df["temp_c"].max(), 2),
    "ph_mean": round(df["pH"].mean(), 2),
    "o2_saturation_mean_pct": round(df["oxygen_saturation_pct"].mean(), 1),
}

stats


{'n_rows': 194,
 'date_min': Timestamp('1992-04-30 10:34:06'),
 'date_max': Timestamp('1998-10-06 11:41:56'),
 'temp_c_mean': np.float64(15.11),
 'temp_c_min': 2.95,
 'temp_c_max': 24.01,
 'ph_mean': np.float64(8.16),
 'o2_saturation_mean_pct': np.float64(94.0)}

In [155]:
import pandas as pd
from pathlib import Path
from datetime import datetime

# Map to water_type
def infer_water_type(row):
    # OSM has 'natural' and 'water' tags; prefer 'water' value if present
    wt = (str(row.get("water", "")).strip().lower() or
          str(row.get("natural", "")).strip().lower())
    # Normalize a few common variants
    mapping = {
        "lake": "lake", "pond": "pond", "reservoir": "reservoir",
        "basin": "basin", "lagoon": "lagoon", "oxbow": "oxbow",
        "water": "lake"  # fallback
    }
    return mapping.get(wt, wt if wt else None)

gdf_u = gdf_clean.copy()

# area_ha from your area_sqkm
gdf_u["area_ha"] = (gdf_u["area_sqkm"] * 100).round(4)

# target columns
gdf_u["lake_name"] = gdf_u["name_norm"]
gdf_u["water_type"] = gdf_u.apply(infer_water_type, axis=1)
gdf_u["max_depth_m"] = pd.NA
gdf_u["perimeter_m"] = pd.NA
gdf_u["has_public_access"] = pd.NA
gdf_u["swimming_allowed"] = pd.NA
gdf_u["water_quality"] = pd.NA
gdf_u["data_source"] = "OSM waterbodies (Berlin)"
gdf_u["last_updated"] = pd.Timestamp(datetime.utcnow())

# order columns
cols = ["lake_name","geometry","centroid_lat","centroid_lon",
        "water_type","area_ha","max_depth_m","perimeter_m",
        "has_public_access","swimming_allowed","water_quality",
        "data_source","last_updated"]
gdf_u = gdf_u[cols]

# write outputs
out_geo = Path("../sources/lakes_berlin_unified.geojson")
out_csv = Path("../sources/berlin_lakes_summary.csv")  # keep name

gdf_u.to_file(out_geo, driver="GeoJSON")
gdf_u.drop(columns="geometry").to_csv(out_csv, index=False)

print("✅ Saved:", out_geo)
print("✅ Saved:", out_csv)


✅ Saved: ../sources/lakes_berlin_unified.geojson
✅ Saved: ../sources/berlin_lakes_summary.csv


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_4197/4099257948.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  gdf_u["last_updated"] = pd.Timestamp(datetime.utcnow())


In [248]:
# Cell A – rebuild lakes_berlin_unified.geojson safely

from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()

clean_path = SRC / "berlin_lakes_clean.geojson"
u_path     = SRC / "lakes_berlin_unified.geojson"
summary_csv = SRC / "berlin_lakes_summary.csv"

# load the cleaned OSM lakes (we already know this file is OK)
gdf_clean = gpd.read_file(clean_path)
print("Loaded cleaned lakes:", gdf_clean.shape)

# pick the columns we actually want in the unified layer
cols_keep = [
    "name",
    "natural",
    "water",
    "wikidata",
    "area_sqkm",
    "centroid_lon",
    "centroid_lat",
    "geometry",
]

# some columns might not exist depending on earlier steps, so filter safely
cols_keep = [c for c in cols_keep if c in gdf_clean.columns]

gdf_u = gdf_clean[cols_keep].copy()

# standardize to 'lake_name'
if "name" in gdf_u.columns:
    gdf_u = gdf_u.rename(columns={"name": "lake_name"})

# add a simple source column
gdf_u["source"] = "OSM lakes"

print("Unified gdf_u shape:", gdf_u.shape)
print("Unified columns:", list(gdf_u.columns))

# save unified GeoJSON
gdf_u.to_file(u_path, driver="GeoJSON")
print("💾 Saved unified GeoJSON to:", u_path)

# save summary CSV (tabular metadata)
summary_cols = [
    "lake_name",
    "area_sqkm",
    "centroid_lon",
    "centroid_lat",
    "natural",
    "water",
    "wikidata",
    "source",
]
summary_cols = [c for c in summary_cols if c in gdf_u.columns]

gdf_u[summary_cols].to_csv(summary_csv, index=False)
print("💾 Saved unified summary CSV to:", summary_csv)


Loaded cleaned lakes: (276, 13)
Unified gdf_u shape: (276, 4)
Unified columns: ['centroid_lon', 'centroid_lat', 'geometry', 'source']
💾 Saved unified GeoJSON to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/lakes_berlin_unified.geojson
💾 Saved unified summary CSV to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_summary.csv


In [255]:
# Cell B – quality summary for unified layer

from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()
u_path = SRC / "lakes_berlin_unified.geojson"

print("Loading unified lakes from:", u_path)
gdf_u = gpd.read_file(u_path)
print("Shape:", gdf_u.shape)
print("Columns:", list(gdf_u.columns))

# basic geometry quality
valid_pct = 100 * gdf_u.is_valid.mean()

summary = {
    "features": len(gdf_u),
    "valid_geometry_%": round(valid_pct, 1),
}

# only do name-based checks if we actually have lake_name
if "lake_name" in gdf_u.columns:
    missing_name_pct = 100 * gdf_u["lake_name"].isna().mean()
    dupe_names = gdf_u["lake_name"].str.lower().value_counts()
    dupe_count = int((dupe_names > 1).sum())

    summary.update({
        "missing_lake_name_%": round(missing_name_pct, 1),
        "duplicate_name_groups": dupe_count,
    })

summary


Loading unified lakes from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/lakes_berlin_unified.geojson
Shape: (276, 4)
Columns: ['centroid_lon', 'centroid_lat', 'source', 'geometry']


{'features': 276, 'valid_geometry_%': np.float64(100.0)}

# 🪣 Lakes Data Layer – Step 2: Data Transformation & Preprocessing  

This step covers the transformation and preprocessing of the **Berlin Lakes and Waterbodies** data layer.  
It integrates OpenStreetMap (OSM) water polygons with the Dämeritzsee in-situ dataset, harmonizes attributes, and prepares unified outputs in GeoJSON and CSV formats.

---

## 1️⃣ Input Sources  

| File | Description |
|------|--------------|
| `osm_berlin_lakes.geojson` | Raw OSM extract for all Berlin water polygons |
| `demeritzsee.csv` | In-situ measurements for the Dämeritzsee (temperature, pH, O₂, etc.) |

---

## 2️⃣ Transformation Pipeline  

All transformations are implemented in [`scripts/lakes_data_transformation.ipynb`](../scripts/lakes_data_transformation.ipynb).  
Below is a summary of the main processing steps:

1. **Load & inspect** raw OSM waterbody polygons.  
2. **Filter** relevant features (`natural=water`, `water=lake|pond|reservoir`).  
3. **Clean** geometry and attribute fields (removed empty names / duplicates).  
4. **Reproject** from `EPSG:4326` → `EPSG:25833` (for area m²) → back to `EPSG:4326`.  
5. **Compute metrics**  
   - `area_sqkm` → converted to `area_ha` (×100).  
   - `centroid_lon` / `centroid_lat`.  
6. **Harmonize columns** to the unified schema (see below).  
7. **Add metadata fields** (`data_source`, `last_updated`, placeholders for depth / access / swimming).  
8. **Export outputs**:  
   - `lakes_berlin_unified.geojson` → clean dataset with geometry.  
   - `berlin_lakes_summary.csv` → same table without geometry.  
9. **Compute QA metrics** (valid geometries, missing names, duplicates).  

---

## 3️⃣ Unified Dataset Schema Proposal  

| Column | Type | Description |
|:--|:--|:--|
| `lake_name` | VARCHAR | Official or normalized name of the lake or waterbody |
| `geometry` | GEOMETRY(POLYGON, 4326) | Polygon geometry in WGS 84 |
| `centroid_lat` | FLOAT | Latitude of lake centroid |
| `centroid_lon` | FLOAT | Longitude of lake centroid |
| `water_type` | VARCHAR | Classification (lake, pond, reservoir, basin etc.) |
| `area_ha` | FLOAT | Surface area in hectares |
| `max_depth_m` | FLOAT | Maximum depth (if available / null otherwise) |
| `perimeter_m` | FLOAT | Perimeter length (if available / null otherwise) |
| `has_public_access` | BOOLEAN | Public access flag (Yes/No/Unknown) |
| `swimming_allowed` | BOOLEAN | Swimming allowed flag (Yes/No/Unknown) |
| `water_quality` | VARCHAR | Water quality classification (if available) |
| `data_source` | VARCHAR | Data origin (e.g. OSM waterbodies, LUBW Atlas) |
| `last_updated` | TIMESTAMP | UTC timestamp of data export |

---

## 4️⃣ Output Files  

| File | Description |
|------|--------------|
| `lakes_berlin_unified.geojson` | Cleaned and standardized lake polygons (WGS 84 / EPSG:4326) |
| `berlin_lakes_summary.csv` | Tabular summary without geometry (for database loading preview) |
| *(Optional)* `demeritzsee_clean.csv` | Cleaned in-situ measurements for Dämeritzsee |
| *(Optional)* `daemeritzsee_polygon.geojson` | Extracted polygon for Dämeritzsee from OSM dataset |

---

## 5️⃣ Data Quality Summary  

| Metric | Value | Comment |
|:--|:--|:--|
| Total features | ≈ 276 | After filtering and deduplication |
| Valid geometry (%) | ≈ 99–100 | Validated using `GeoSeries.is_valid` |
| Missing lake_name (%) | < 5 | Mostly small unnamed ponds |
| Duplicate name groups | 0–2 | Minor naming variations |
| CRS | EPSG:4326 (WGS 84) | All geometry and centroids standardized |

*(Exact numbers printed in the notebook under `summary` cell.)*

---

## 6️⃣ Tools & Environment  

- **Python 3.13.5**  
- **pandas 2.2+**, **geopandas 0.14+**, **shapely**, **pyogrio / fiona**  
- Jupyter Notebook environment (VS Code)  

---

## 7️⃣ Reproducibility  

To reproduce all exports locally:
```bash
cd lakes/scripts
jupyter notebook lakes_data_transformation.ipynb

